In [ ]:
import sys
import os

!pip install transformer_lens scikit-learn matplotlib tqdm -q

if not os.path.exists('/content/MATS-take-home'):
    !git clone https://github.com/vedantgaur/MATS-take-home.git
else:
    %cd /content/MATS-take-home
    !git pull
    %cd /content

sys.path.append('/content/MATS-take-home')

import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

from data.mess3_generator import Mess3Process, NonErgodicMess3Dataset
from models.train import get_toy_config, train_model
from transformer_lens import HookedTransformer
from analysis.geometry import extract_activations, calculate_cev, plot_cev, plot_2d_pca
from analysis.orthogonality import compare_all_processes

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
p1 = Mess3Process(alpha=0.60, x=0.15)
p2 = Mess3Process(alpha=0.79, x=0.11)
p3 = Mess3Process(alpha=0.60, x=0.50)

num_train_samples = 4000
seq_length = 16
dataset = NonErgodicMess3Dataset(
    num_samples=num_train_samples, 
    seq_length=seq_length, 
    processes=[p1, p2, p3]
)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

val_dataset = NonErgodicMess3Dataset(
    num_samples=1000, 
    seq_length=seq_length, 
    processes=[p1, p2, p3]
)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [ ]:
cfg = get_toy_config(vocab_size=3, d_model=64, n_ctx=seq_length)
model = HookedTransformer(cfg).to(device)

print("Starting training...")
loss_history = train_model(model, train_loader, epochs=15, lr=5e-4)

plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross Entropy Loss")
plt.show()

In [ ]:
from analysis.geometry import extract_all_layers

print("Extracting activations across layers...")
acts_by_layer, labels = extract_all_layers(model, val_loader)

n_layers = len(acts_by_layer)

for layer_idx in range(n_layers):
    print(f"\n{'='*40}")
    print(f"ANALYZING LAYER {layer_idx}")
    print(f"{'='*40}")
    
    acts = acts_by_layer[layer_idx]
    
    pca_model, cev = calculate_cev(acts)
    dims_needed = plot_cev(cev, threshold=0.95)
    print(f"[Layer {layer_idx}] Dimensions required for 95% variance: {dims_needed}")
    
    # In Layer 0, we expect these top components to cluster purely by process ID.
    plot_2d_pca(acts, labels, pca_model, pc_x=0, pc_y=1)
    
    # In later layers, the 2-simplices should become visible.
    plot_2d_pca(acts, labels, pca_model, pc_x=2, pc_y=3)
    
    print(f"[Layer {layer_idx}] Calculating pairwise subspace overlaps...")
    overlap_matrix = compare_all_processes(acts, labels, k_dims=2)
    
    plt.figure(figsize=(6, 5))
    plt.imshow(overlap_matrix, cmap='magma', vmin=0, vmax=1)
    plt.colorbar(label='Overlap Score (0=Orthogonal, 1=Parallel)')
    plt.xticks(ticks=[0, 1, 2], labels=['Process 1', 'Process 2', 'Process 3'])
    plt.yticks(ticks=[0, 1, 2], labels=['Process 1', 'Process 2', 'Process 3'])
    plt.title(f"Layer {layer_idx} Subspace Overlap")
    plt.show()
    
    print(f"[Layer {layer_idx}] Overlap Matrix:")
    print(np.round(overlap_matrix, 3))